# HFENN 质量门控分析

## 目标
1. 检测"死信号"窗口（pulse_std ≈ 0）
2. 对比三种评估设置：Full / Clean / Full+Fallback
3. 输出各设置下的 R²/MAE/RMSE + 残差ACF + 同号段统计

## 质量检测规则
- **Level-1（硬判定）**：`pulse_range < 1e-8` 或 `pulse_std < 1e-8`
- **Level-2（软判定）**：`flat_ratio > 0.98` 且 `pulse_std < p1(train)`

In [ ]:
# === Cell 1: 导入依赖 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("依赖导入完成")

In [ ]:
# === Cell 2: 从Affine实验notebook加载必要变量 ===
# 注意：需要先运行 HFENN_affine_calibration_exp.ipynb 的所有Cell
# 这里我们直接使用那个notebook中已经计算好的变量

print("="*70)
print("质量门控分析 - 数据加载")
print("="*70)

# 检查必要变量是否存在
required_vars = [
    'X_pulse_all', 'y_all', 'window_idx_all',
    'offline_oracle_test_indices', 'offline_oracle_calib_indices',
    'y_pred_base_all', 'y_pred_ft_all',
    'offline_oracle_results'
]

missing_vars = [v for v in required_vars if v not in dir()]
if missing_vars:
    print(f"⚠️ 缺少变量: {missing_vars}")
    print("请先运行 HFENN_affine_calibration_exp.ipynb 的所有Cell")
else:
    print("✅ 所有必要变量已加载")
    print(f"  总窗口数: {len(y_all)}")
    print(f"  测试窗口数: {len(offline_oracle_test_indices)}")

In [ ]:
# === Cell 3: 计算窗口质量特征 ===
print("="*70)
print("Step 1: 计算窗口质量特征")
print("="*70)

def compute_window_quality(X_pulse):
    """
    计算每个窗口的质量特征
    
    Returns:
        quality_df: DataFrame with quality metrics for each window
    """
    n_windows = len(X_pulse)
    
    pulse_std = np.zeros(n_windows)
    pulse_range = np.zeros(n_windows)
    flat_ratio = np.zeros(n_windows)
    
    for i in range(n_windows):
        signal = X_pulse[i]
        pulse_std[i] = np.std(signal)
        pulse_range[i] = np.ptp(signal)  # max - min
        
        # flat_ratio: 相邻差分为0的比例
        diff = np.diff(signal)
        flat_ratio[i] = np.mean(np.abs(diff) < 1e-10)
    
    quality_df = pd.DataFrame({
        'window_idx': np.arange(n_windows),
        'pulse_std': pulse_std,
        'pulse_range': pulse_range,
        'flat_ratio': flat_ratio
    })
    
    return quality_df

# 计算所有窗口的质量特征
quality_df = compute_window_quality(X_pulse_all)

print(f"\n质量特征统计:")
print(quality_df[['pulse_std', 'pulse_range', 'flat_ratio']].describe())

In [ ]:
# === Cell 4: Level-1 死信号检测 ===
print("="*70)
print("Step 2: Level-1 死信号检测")
print("="*70)

# Level-1 规则：pulse_range < 1e-8 或 pulse_std < 1e-8
LEVEL1_THRESHOLD = 1e-8

quality_df['is_dead_signal'] = (
    (quality_df['pulse_range'] < LEVEL1_THRESHOLD) | 
    (quality_df['pulse_std'] < LEVEL1_THRESHOLD)
)

dead_windows = quality_df[quality_df['is_dead_signal']]
n_dead = len(dead_windows)

print(f"\n【Level-1 检测结果】")
print(f"  阈值: pulse_std < {LEVEL1_THRESHOLD} 或 pulse_range < {LEVEL1_THRESHOLD}")
print(f"  死信号窗口数: {n_dead} / {len(quality_df)} ({100*n_dead/len(quality_df):.2f}%)")

if n_dead > 0:
    print(f"\n  死信号窗口详情:")
    print(dead_windows[['window_idx', 'pulse_std', 'pulse_range', 'flat_ratio']].to_string())

In [ ]:
# === Cell 5: 识别测试集中的死信号窗口 ===
print("="*70)
print("Step 3: 测试集死信号分析")
print("="*70)

# 获取测试集窗口的质量信息
test_idx = offline_oracle_test_indices
test_quality = quality_df.iloc[test_idx].copy()
test_quality['test_local_idx'] = np.arange(len(test_idx))

# 测试集中的死信号
test_dead = test_quality[test_quality['is_dead_signal']]
n_test_dead = len(test_dead)

print(f"\n【测试集死信号】")
print(f"  测试集总窗口: {len(test_idx)}")
print(f"  死信号窗口: {n_test_dead} ({100*n_test_dead/len(test_idx):.2f}%)")

if n_test_dead > 0:
    print(f"\n  死信号窗口详情:")
    for _, row in test_dead.iterrows():
        print(f"    全局索引={int(row['window_idx'])}, 测试集内索引={int(row['test_local_idx'])}, "
              f"pulse_std={row['pulse_std']:.2e}, pulse_range={row['pulse_range']:.2e}")

# 保存clean测试集索引
dead_global_indices = set(test_dead['window_idx'].values)
clean_test_mask = ~test_quality['is_dead_signal'].values
clean_test_local_indices = np.where(clean_test_mask)[0]

print(f"\n  Clean测试集窗口数: {len(clean_test_local_indices)}")

In [ ]:
# === Cell 6: 准备三种评估设置的数据 ===
print("="*70)
print("Step 4: 准备三种评估设置")
print("="*70)

# 获取Affine(Base)的预测值
y_pred_affine_base_test = offline_oracle_results['predictions']['y_pred_affine_base_test']
y_true_test = y_all[test_idx]

# === Setting 1: Full (原样) ===
y_true_full = y_true_test
y_pred_full = y_pred_affine_base_test

# === Setting 2: Clean (过滤死信号) ===
y_true_clean = y_true_test[clean_test_local_indices]
y_pred_clean = y_pred_affine_base_test[clean_test_local_indices]

# === Setting 3: Full + Fallback (死信号用y[t-1]) ===
y_pred_fallback = y_pred_affine_base_test.copy()

# 对死信号窗口应用持久性fallback
dead_local_indices = np.where(~clean_test_mask)[0]
for idx in dead_local_indices:
    if idx > 0:
        # 使用前一个窗口的预测值
        y_pred_fallback[idx] = y_pred_fallback[idx - 1]
    else:
        # 如果是第一个窗口，保持原值（或可以用其他策略）
        pass

y_true_fallback = y_true_test

print(f"\n【三种设置数据准备完成】")
print(f"  Full:     {len(y_true_full)} 窗口")
print(f"  Clean:    {len(y_true_clean)} 窗口 (过滤了 {len(y_true_full) - len(y_true_clean)} 个死信号)")
print(f"  Fallback: {len(y_true_fallback)} 窗口 (对 {len(dead_local_indices)} 个死信号应用持久性)")

In [ ]:
# === Cell 7: 计算三种设置的指标 ===
print("="*70)
print("Step 5: 计算 R²/MAE/RMSE")
print("="*70)

def compute_metrics(y_true, y_pred):
    """计算回归指标"""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'R2': r2, 'MAE': mae, 'RMSE': rmse}

# 计算三种设置的指标
metrics_full = compute_metrics(y_true_full, y_pred_full)
metrics_clean = compute_metrics(y_true_clean, y_pred_clean)
metrics_fallback = compute_metrics(y_true_fallback, y_pred_fallback)

# 汇总表格
metrics_df = pd.DataFrame({
    'Setting': ['Full', 'Clean', 'Full+Fallback'],
    'N_Windows': [len(y_true_full), len(y_true_clean), len(y_true_fallback)],
    'R²': [metrics_full['R2'], metrics_clean['R2'], metrics_fallback['R2']],
    'MAE': [metrics_full['MAE'], metrics_clean['MAE'], metrics_fallback['MAE']],
    'RMSE': [metrics_full['RMSE'], metrics_clean['RMSE'], metrics_fallback['RMSE']]
})

print("\n【Offline-oracle + Affine(Base) 指标对比】")
print(metrics_df.to_string(index=False))

# 计算提升
print(f"\n【Clean vs Full 提升】")
print(f"  ΔR²:   {metrics_clean['R2'] - metrics_full['R2']:+.4f}")
print(f"  ΔMAE:  {metrics_clean['MAE'] - metrics_full['MAE']:+.4f}")
print(f"  ΔRMSE: {metrics_clean['RMSE'] - metrics_full['RMSE']:+.4f}")

print(f"\n【Fallback vs Full 提升】")
print(f"  ΔR²:   {metrics_fallback['R2'] - metrics_full['R2']:+.4f}")
print(f"  ΔMAE:  {metrics_fallback['MAE'] - metrics_full['MAE']:+.4f}")
print(f"  ΔRMSE: {metrics_fallback['RMSE'] - metrics_full['RMSE']:+.4f}")

In [ ]:
# === Cell 8: 计算残差ACF ===
print("="*70)
print("Step 6: 残差自相关分析 (ACF)")
print("="*70)

def compute_acf(residuals, max_lag=30):
    """计算自相关函数"""
    max_lag = min(max_lag, len(residuals) // 4)
    acf_values = [1.0]  # lag=0
    
    for lag in range(1, max_lag + 1):
        if len(residuals) > lag:
            corr, _ = pearsonr(residuals[:-lag], residuals[lag:])
            acf_values.append(corr)
        else:
            acf_values.append(np.nan)
    
    return np.array(acf_values)

# 计算三种设置的残差
residuals_full = y_true_full - y_pred_full
residuals_clean = y_true_clean - y_pred_clean
residuals_fallback = y_true_fallback - y_pred_fallback

# 计算ACF
acf_full = compute_acf(residuals_full)
acf_clean = compute_acf(residuals_clean)
acf_fallback = compute_acf(residuals_fallback)

# 95%置信区间
ci_full = 1.96 / np.sqrt(len(residuals_full))
ci_clean = 1.96 / np.sqrt(len(residuals_clean))
ci_fallback = 1.96 / np.sqrt(len(residuals_fallback))

print("\n【残差ACF对比 (Lag 1-10)】")
print(f"{'Lag':<6} {'Full':<12} {'Clean':<12} {'Fallback':<12}")
print("-" * 45)
for lag in range(1, 11):
    print(f"{lag:<6} {acf_full[lag]:<12.4f} {acf_clean[lag]:<12.4f} {acf_fallback[lag]:<12.4f}")

print(f"\n【显著性阈值 (95% CI)】")
print(f"  Full:     ±{ci_full:.4f}")
print(f"  Clean:    ±{ci_clean:.4f}")
print(f"  Fallback: ±{ci_fallback:.4f}")

# 统计显著自相关的滞后数
sig_full = sum(1 for i, acf in enumerate(acf_full[1:11], 1) if abs(acf) > ci_full)
sig_clean = sum(1 for i, acf in enumerate(acf_clean[1:11], 1) if abs(acf) > ci_clean)
sig_fallback = sum(1 for i, acf in enumerate(acf_fallback[1:11], 1) if abs(acf) > ci_fallback)

print(f"\n【Lag 1-10 中显著自相关的个数】")
print(f"  Full:     {sig_full}/10")
print(f"  Clean:    {sig_clean}/10")
print(f"  Fallback: {sig_fallback}/10")

In [ ]:
# === Cell 9: 连续同号段统计 ===
print("="*70)
print("Step 7: 连续同号段统计")
print("="*70)

def compute_run_stats(residuals):
    """计算连续同号段统计"""
    signs = np.sign(residuals)
    
    runs = []
    current_sign = signs[0]
    current_length = 1
    
    for i in range(1, len(signs)):
        if signs[i] == current_sign:
            current_length += 1
        else:
            runs.append(current_length)
            current_sign = signs[i]
            current_length = 1
    runs.append(current_length)
    
    n = len(residuals)
    expected_runs = (2 * n - 1) / 3
    expected_run_length = 1.5
    
    return {
        'n_runs': len(runs),
        'mean_run_length': np.mean(runs),
        'max_run_length': max(runs),
        'expected_runs': expected_runs,
        'expected_run_length': expected_run_length,
        'runs_ratio': len(runs) / expected_runs,
        'length_ratio': np.mean(runs) / expected_run_length
    }

# 计算三种设置的同号段统计
runs_full = compute_run_stats(residuals_full)
runs_clean = compute_run_stats(residuals_clean)
runs_fallback = compute_run_stats(residuals_fallback)

print("\n【连续同号段统计对比】")
print(f"{'指标':<20} {'Full':<12} {'Clean':<12} {'Fallback':<12}")
print("-" * 60)
print(f"{'总段数':<20} {runs_full['n_runs']:<12} {runs_clean['n_runs']:<12} {runs_fallback['n_runs']:<12}")
print(f"{'平均段长':<20} {runs_full['mean_run_length']:<12.2f} {runs_clean['mean_run_length']:<12.2f} {runs_fallback['mean_run_length']:<12.2f}")
print(f"{'最大段长':<20} {runs_full['max_run_length']:<12} {runs_clean['max_run_length']:<12} {runs_fallback['max_run_length']:<12}")
print(f"{'期望段数':<20} {runs_full['expected_runs']:<12.1f} {runs_clean['expected_runs']:<12.1f} {runs_fallback['expected_runs']:<12.1f}")
print(f"{'段数比(实际/期望)':<20} {runs_full['runs_ratio']:<12.2f} {runs_clean['runs_ratio']:<12.2f} {runs_fallback['runs_ratio']:<12.2f}")
print(f"{'段长比(实际/期望)':<20} {runs_full['length_ratio']:<12.2f} {runs_clean['length_ratio']:<12.2f} {runs_fallback['length_ratio']:<12.2f}")

print("\n【解读】")
print("  段数比 < 1: 实际段数少于随机期望，说明残差有趋势性")
print("  段长比 > 1: 实际段长大于随机期望，说明残差有持续性偏差")

In [ ]:
# === Cell 10: 可视化对比 ===
print("="*70)
print("Step 8: 可视化")
print("="*70)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('质量门控分析: Full vs Clean vs Fallback', fontsize=14)

settings = ['Full', 'Clean', 'Fallback']
residuals_list = [residuals_full, residuals_clean, residuals_fallback]
acf_list = [acf_full, acf_clean, acf_fallback]
ci_list = [ci_full, ci_clean, ci_fallback]

# 第一行：残差时间序列
for i, (name, res) in enumerate(zip(settings, residuals_list)):
    ax = axes[0, i]
    ax.plot(res, 'b-', alpha=0.7, linewidth=0.8)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=1)
    ax.fill_between(range(len(res)), res, 0, 
                    where=(res > 0), color='green', alpha=0.3)
    ax.fill_between(range(len(res)), res, 0, 
                    where=(res < 0), color='red', alpha=0.3)
    ax.set_title(f'{name} 残差 (N={len(res)})')
    ax.set_xlabel('窗口索引')
    ax.set_ylabel('残差')
    ax.grid(True, alpha=0.3)

# 第二行：ACF
for i, (name, acf, ci) in enumerate(zip(settings, acf_list, ci_list)):
    ax = axes[1, i]
    lags = range(len(acf))
    ax.bar(lags, acf, color='steelblue', edgecolor='black', width=0.8)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axhline(y=ci, color='r', linestyle='--', alpha=0.7)
    ax.axhline(y=-ci, color='r', linestyle='--', alpha=0.7)
    ax.set_title(f'{name} ACF')
    ax.set_xlabel('Lag')
    ax.set_ylabel('自相关')
    ax.set_xlim(-0.5, 30.5)

plt.tight_layout()
plt.savefig('HFENN_quality_gate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 图片已保存: HFENN_quality_gate_comparison.png")

In [ ]:
# === Cell 11: 总结报告 ===
print("="*70)
print("质量门控分析总结")
print("="*70)

print("\n【1. 死信号检测】")
print(f"  Level-1阈值: pulse_std < 1e-8 或 pulse_range < 1e-8")
print(f"  全部窗口: {len(quality_df)} 个")
print(f"  死信号窗口: {n_dead} 个 ({100*n_dead/len(quality_df):.2f}%)")
print(f"  测试集死信号: {n_test_dead} 个 ({100*n_test_dead/len(test_idx):.2f}%)")

print("\n【2. 指标对比 (Offline-oracle + Affine(Base))】")
print(metrics_df.to_string(index=False))

print("\n【3. 残差时间结构】")
print(f"  {'设置':<12} {'Lag-1 ACF':<12} {'平均段长':<12} {'段长比':<12}")
print(f"  {'-'*48}")
print(f"  {'Full':<12} {acf_full[1]:<12.4f} {runs_full['mean_run_length']:<12.2f} {runs_full['length_ratio']:<12.2f}")
print(f"  {'Clean':<12} {acf_clean[1]:<12.4f} {runs_clean['mean_run_length']:<12.2f} {runs_clean['length_ratio']:<12.2f}")
print(f"  {'Fallback':<12} {acf_fallback[1]:<12.4f} {runs_fallback['mean_run_length']:<12.2f} {runs_fallback['length_ratio']:<12.2f}")

print("\n【4. 结论】")

# 判断Clean是否有显著提升
r2_improvement = metrics_clean['R2'] - metrics_full['R2']
if r2_improvement > 0.01:
    print(f"  ✅ Clean设置R²提升显著 (+{r2_improvement:.4f})")
    print(f"     建议: 主结果使用Clean test，附加报告Full test")
else:
    print(f"  ⚠️ Clean设置R²提升有限 (+{r2_improvement:.4f})")
    print(f"     死信号对整体指标影响较小")

# 判断残差结构是否减弱
acf_reduction = acf_full[1] - acf_clean[1]
if acf_reduction > 0.05:
    print(f"  ✅ Clean设置残差自相关减弱 (Lag-1 ACF: {acf_full[1]:.4f} → {acf_clean[1]:.4f})")
else:
    print(f"  ⚠️ Clean设置残差自相关变化不大 (Lag-1 ACF: {acf_full[1]:.4f} → {acf_clean[1]:.4f})")
    print(f"     残差时间结构主要不是由死信号引起")

# STM建议
if acf_clean[1] > 0.5:
    print(f"\n  📌 即使过滤死信号后，Lag-1 ACF仍然很高 ({acf_clean[1]:.4f})")
    print(f"     强烈建议使用STM方法来利用残差的时间结构")
elif acf_clean[1] > 0.3:
    print(f"\n  📌 过滤后Lag-1 ACF中等 ({acf_clean[1]:.4f})")
    print(f"     可以考虑STM方法，但收益可能有限")
else:
    print(f"\n  📌 过滤后Lag-1 ACF较低 ({acf_clean[1]:.4f})")
    print(f"     残差接近白噪声，STM可能不会带来显著提升")

In [ ]:
# === Cell 12: 保存结果 ===
print("="*70)
print("保存结果")
print("="*70)

# 保存指标对比
metrics_df.to_csv('HFENN_quality_gate_metrics.csv', index=False)
print("✅ 指标对比已保存: HFENN_quality_gate_metrics.csv")

# 保存ACF对比
acf_df = pd.DataFrame({
    'Lag': range(len(acf_full)),
    'ACF_Full': acf_full,
    'ACF_Clean': acf_clean,
    'ACF_Fallback': acf_fallback
})
acf_df.to_csv('HFENN_quality_gate_acf.csv', index=False)
print("✅ ACF对比已保存: HFENN_quality_gate_acf.csv")

# 保存同号段统计
runs_df = pd.DataFrame({
    'Setting': ['Full', 'Clean', 'Fallback'],
    'N_Runs': [runs_full['n_runs'], runs_clean['n_runs'], runs_fallback['n_runs']],
    'Mean_Run_Length': [runs_full['mean_run_length'], runs_clean['mean_run_length'], runs_fallback['mean_run_length']],
    'Max_Run_Length': [runs_full['max_run_length'], runs_clean['max_run_length'], runs_fallback['max_run_length']],
    'Runs_Ratio': [runs_full['runs_ratio'], runs_clean['runs_ratio'], runs_fallback['runs_ratio']],
    'Length_Ratio': [runs_full['length_ratio'], runs_clean['length_ratio'], runs_fallback['length_ratio']]
})
runs_df.to_csv('HFENN_quality_gate_runs.csv', index=False)
print("✅ 同号段统计已保存: HFENN_quality_gate_runs.csv")

# 保存死信号窗口列表
dead_windows.to_csv('HFENN_dead_signal_windows.csv', index=False)
print("✅ 死信号窗口列表已保存: HFENN_dead_signal_windows.csv")

print("\n✅ 所有结果已保存")